In [ ]:
%pip install "transformers==4.57.6" "datasets==5.0.1" \
    "accelerate==1.15.0" "pythainlp==5.3.8" \
    "sentencepiece==0.2.1" --quiet

# Solution — Module M3: Thai sentiment fine-tune

เฉลยครบทุกข้อของ `exercise.ipynb` (ผู้สอนเปิดเผยหลังจบแล็บ) — ใช้ `N_TRAIN = 4_000`, `N_EPOCHS = 2` เดียวกับแบบฝึกหัด ตัวเลขที่พิมพ์ในคำตอบเป็นค่าจากรอบหนึ่ง (seed 42) — รอบของคุณจะใกล้เคียงกันแต่ไม่เท่ากันเป๊ะ

**Dataset:** Wisesight Sentiment — <https://huggingface.co/datasets/pythainlp/wisesight_sentiment>, **License: CC0 1.0** · **Model:** mBERT (`google-bert/bert-base-multilingual-cased`), **License: Apache-2.0**

## 0. เตรียมข้อมูล + เทรน (ให้มาแล้ว — รันผ่าน)

โค้ดชุดนี้คือส่วนสำคัญของ book ย่อมาไว้ด้วยกัน: โหลด Wisesight → subsample 4,000 ข้อความ → tokenize (รันเซลล์นี้ผ่านก่อน)

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

transformers.set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch",
      torch.__version__)
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: ไม่มี GPU — บน CPU จะช้ามาก ให้รันบน Kaggle GPU")

N_TRAIN = 4_000
N_EPOCHS = 2

import matplotlib.pyplot as plt  # noqa: E402

from datasets import load_dataset  # noqa: E402

ds = load_dataset("pythainlp/wisesight_sentiment")
LABELS = ds["train"].features["category"].names
print(ds)
print("labels:", LABELS)

from transformers import AutoTokenizer  # noqa: E402

CHECKPOINT = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print(tokenizer.tokenize("การสอบเทียบเครื่องมือวัด")[:14])

from sklearn.model_selection import train_test_split  # noqa: E402

# subsample แบบ stratified ตาม class — จบเร็วขึ้นแต่สัดส่วนคลาสเดิม
y_all = np.array(ds["train"]["category"])
idx_sub, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=N_TRAIN,
    stratify=y_all,
    random_state=SEED,
)
idx_probe, _ = train_test_split(
    np.arange(len(y_all)),
    train_size=1_000,
    stratify=y_all,
    random_state=SEED,
)

splits = {
    "train": ds["train"].select(idx_sub),
    "train_probe": ds["train"].select(idx_probe),
    "val": ds["validation"],
    "test": ds["test"],
}


def tokenize(batch):
    return tokenizer(batch["texts"], truncation=True, max_length=64)


tok = {}
for name, d in splits.items():
    d = d.rename_column("category", "labels")
    tok[name] = d.map(tokenize, batched=True)
print({name: len(d) for name, d in tok.items()})

ต่อ: โหลด mBERT, fine-tune 4_000 ข้อความ × 2 epochs (ใช้เวลา ~3–5 นาทีบน T4 — รอจนจบ) แล้วทำนายผลบน test set — จะได้ `trainer`, `tok`, `y_true`, `y_pred`, `texts_test` ไว้ใช้ทุกข้อ

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

id2label = {i: name for i, name in enumerate(LABELS)}
label2id = {name: i for i, name in enumerate(LABELS)}
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="m3_wisesight",
    seed=SEED,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=N_EPOCHS,
    weight_decay=0.01,
    fp16=DEVICE == "cuda",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}


trainer = Trainer(
    model,
    args,
    train_dataset=tok["train"],
    eval_dataset={
        "train_probe": tok["train_probe"],
        "val": tok["val"],
    },
    data_collator=DataCollatorWithPadding(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

from sklearn.metrics import classification_report  # noqa: E402

pred_out = trainer.predict(tok["test"])
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids
texts_test = ds["test"]["texts"]
print(classification_report(
    y_true, y_pred,
    labels=range(len(LABELS)), target_names=LABELS, digits=3,
    zero_division=0,
))

loss ราย epoch ของการเทรนนี้ (ไว้อ้างอิงข้อ 2):

In [ ]:
history = trainer.state.log_history
epochs = [h["epoch"] for h in history if "loss" in h]
train_loss = [h["loss"] for h in history if "loss" in h]
probe_loss = [
    h["eval_train_probe_loss"] for h in history
    if "eval_train_probe_loss" in h
]
val_loss = [
    h["eval_val_loss"] for h in history if "eval_val_loss" in h
]
val_acc = [
    h["eval_val_accuracy"] for h in history
    if "eval_val_accuracy" in h
]
print("train loss ราย epoch :", [round(x, 4) for x in train_loss])
print("val loss   ราย epoch :", [round(x, 4) for x in val_loss])
print("val accuracy ราย epoch:", [round(x, 4) for x in val_acc])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, train_loss, "o-", label="train loss")
ax.plot(epochs, probe_loss, "s--", label="train_probe loss")
ax.plot(epochs, val_loss, "^--", label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("loss ลดลงราย epoch ระหว่าง fine-tune")
ax.legend()
plt.show()

และเครื่องมือ embedding ที่ให้มาแล้ว — ฟังก์ชัน `embed()` + เวกเตอร์ `E` ของตัวอย่าง test 500 ตัวอย่าง (ใช้กับข้อ 4):

In [ ]:
from sklearn.decomposition import PCA  # noqa: E402

rng = np.random.default_rng(SEED)
emb_idx = rng.choice(len(y_true), size=500, replace=False)
emb_texts = [texts_test[i] for i in emb_idx]
emb_labels = [LABELS[y_true[i]] for i in emb_idx]


MODEL_DEV = next(trainer.model.parameters()).device


@torch.no_grad()
def embed(texts):
    """เวกเตอร์ [CLS] จากชั้นสุดท้ายของโมเดลที่เทรนแล้ว."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    ).to(MODEL_DEV)
    out = trainer.model(**enc, output_hidden_states=True)
    cls = out.hidden_states[-1][:, 0]
    return cls.float().cpu().numpy()


E = embed(emb_texts)
pca = PCA(n_components=2)
xy = pca.fit_transform(E)

fig, ax = plt.subplots(figsize=(6, 5))
for ci, cname in enumerate(LABELS):
    m = [j for j, y in enumerate(emb_labels) if y == cname]
    ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.5, label=cname)
ax.set_title("embedding (CLS) ของ 500 ข้อความ — PCA 2D")
ax.legend()
plt.show()

## ข้อ 1 — ตัดคำประโยคของตัวเอง

เฉลยใช้ประโยคงานเมโทรโลยี 3 ประโยค — คุณใช้ประโยคของตัวเองก็ได้ ข้อสรุปเชิงวิเคราะห์เหมือนกัน: การตัดคำไม่มีคำตอบสากล 100%

In [ ]:
from pythainlp.tokenize import word_tokenize  # noqa: E402

my_sentences = [
    "ผลการสอบเทียบเครื่องมือวัดอยู่ในเกณฑ์ที่ยอมรับได้",
    "เครื่องชั่งผิดพลาดบ่อยมากซ่อมกี่ครั้งก็ไม่หาย",
    "สอบถามค่าบริการสอบเทียบน้ำหนักชุดที่ 5 หน่อยครับ",
]

for s in my_sentences:
    print("ประโยค:", s)
    print("  split(' ') :", s.split(" "))
    print("  newmm      :", word_tokenize(s))
    print("  longest    :", word_tokenize(s, engine="longest"))

**คำตอบข้อ 1:** `newmm` (กฎ + พจนานุกรม) กับ `longest` (greedy เลือกคำยาวสุด) ตัดต่างกันตรงความกำกวม เช่น "ยอมรับได้" หรือชื่อเฉพาะ — และ `split(" ")` แทบไม่ช่วยเพราะประโยคไทยเขียนติดกัน ข้อสรุป: การตัดคำไทยคือ "การเดาที่มีหลักการ" ไม่ใช่ความจริงสัมบูรณ์ — โมเดล transformer เลือกทางต่างออกไป (wordpiece) ซึ่งไม่ต้องเดา "คำ" เลย

## ข้อ 2 — class ที่อ่อนที่สุด

In [ ]:
from sklearn.metrics import classification_report  # noqa: E402

report = classification_report(
    y_true, y_pred,
    labels=range(len(LABELS)), target_names=LABELS, digits=3,
    zero_division=0, output_dict=True,
)

weakest = min(LABELS, key=lambda c: report[c]["recall"])
ci_weakest = LABELS.index(weakest)
n_true = int((y_true == ci_weakest).sum())
print(f"class ที่ recall ต่ำสุด: {weakest} "
      f"= {report[weakest]['recall']:.3f} "
      f"(มี {n_true} ตัวอย่างจริงใน test)")

**คำตอบข้อ 2:** จากรอบ 4k/2 epochs (seed 42) class ที่ recall ต่ำสุดคือ `neg` (ราว 0.55–0.65 ขึ้นกับรอบ) — แปลว่า **ข้อความเชิงลบจริง 100 ข้อความ โมเดลจับได้ราว 55–65 ที่เหลือหลุดไปเป็น class อื่น** ถ้าใช้ระบบนี้คัดกรองข้อความเชิงลบจริง ต้องยอมรับความเสี่ยงพลาดราว 1 ใน 3 — ตัวเลขนี้แหละที่ทำให้ recall สำคัญกว่า accuracy สำหรับโจทย์ "หาให้ครบ"

(ถ้ารอบของคุณได้ `q` เป็นตัวอ่อนที่สุดก็ตอบตามตัวเลขของตัวเองได้ — สาเหตุคล้ายกัน: ข้อมูลน้อย + ความกำกวมกับ `neu`)

## ข้อ 3 — Error inspection ของ class ที่อ่อนที่สุด

In [ ]:
wrong_idx = [
    i for i in range(len(y_true))
    if y_true[i] == ci_weakest and y_pred[i] != ci_weakest
]
print(f"ผิดทั้งหมด {len(wrong_idx)} ตัวอย่าง — 5 ตัวอย่างแรก:")
for i in wrong_idx[:5]:
    print(f"   ({LABELS[y_true[i]]} -> {LABELS[y_pred[i]]})",
          texts_test[i][:70].replace("\n", " "))

**คำตอบข้อ 3:** pattern ที่พบซ้ำ ๆ ในตัวอย่างที่พลาด (อ่านจากผลที่พิมพ์):

- การบ่นแบบสุภาพ/อ้อม ๆ ("ของมาช้าไปหน่อยแต่โอเคอยู่") — ไม่มีคำชัดว่าลบ มนุษย์รู้จากบริบท
- คำถามที่ไม่มีเครื่องหมาย ? จึงถูกตอบเป็น `neu`
- ข้อความสั้นไม่มีบริบท ("ได้ครับ", "จริงเหรอ") — ตีความได้หลายทาง ขึ้นกับน้ำเสียงที่ข้อความไม่มี
- สะกดผิด/คำย่อ/อิโมจิเยอะ — wordpiece แทนได้แต่ความหมายจาง

## ข้อ 4 — embedding + kNN กับประโยคของตัวเอง

In [ ]:
from sklearn.neighbors import NearestNeighbors  # noqa: E402

my_text = "เครื่องมือวัดของเราค่าเบี่ยงเบนมาตรฐานต่ำมากใช้ได้ดี"
E_my = embed([my_text])
nn2 = NearestNeighbors(n_neighbors=5).fit(E)
D_my, I_my = nn2.kneighbors(E_my)

for rank in range(5):
    j = I_my[0][rank]
    print(f"  #{rank + 1} dist={D_my[0][rank]:.3f} "
          f"[{emb_labels[j]}]",
          emb_texts[j][:55].replace("\n", " "))

**คำตอบข้อ 4:** เพื่อนบ้านที่ได้มักเป็นข้อความที่ "อารมณ์คล้ายกัน" (สำหรับประโยคเชิงบวกของเฉลย: เพื่อนบ้านส่วนใหญ่ label `neu`/`pos` พูดถึงคุณภาพ/ความพอใจ) — สมเหตุสมผล สังเกตว่า dist อันดับ 1 มักเล็กกว่าอันดับถัดไปชัด ถ้าเพื่อนบ้าน label เดียวกันหมด = embedding แยกกลุ่มอารมณ์ได้จริง ถ้าปนกันมาก = ประโยคของคุณอยู่เขตเส้นแบ่ง (เช่น กลาง ๆ neu กับ pos)

## ข้อ 5 — zero-shot กับประโยคของตัวเอง

โหลดโมเดล zero-shot (ดาวน์โหลด ~1.1 GB ครั้งแรก):

In [ ]:
ZS_CHECKPOINT = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
zs_tok = AutoTokenizer.from_pretrained(ZS_CHECKPOINT)
zs_model = AutoModelForSequenceClassification.from_pretrained(
    ZS_CHECKPOINT,
).to(DEVICE)

# XLM-T ทำนาย 3 class: 0=negative 1=neutral 2=positive
# Wisesight เรียง 0=pos 1=neu 2=neg 3=q — ตารางแปลงดังนี้
ZS_MAP = {0: 2, 1: 1, 2: 0}


@torch.no_grad()
def zero_shot(texts):
    """ทำนายด้วย XLM-T แล้วแปลง id ให้ตรงลำดับ LABELS ของ Wisesight."""
    logits = []
    for start in range(0, len(texts), 32):
        enc = zs_tok(
            texts[start:start + 32],
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt",
        ).to(DEVICE)
        out = zs_model(**enc)
        logits.append(out.logits.float().cpu())
    return torch.cat(logits).argmax(-1).numpy()

In [ ]:
@torch.no_grad()
def predict_ft(texts):
    """ทำนายด้วย mBERT ที่เทรนแล้ว — id เดียวกับ LABELS."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    ).to(MODEL_DEV)
    out = trainer.model(**enc)
    return out.logits.argmax(-1).cpu().numpy()


zs_pred_my = zero_shot(my_sentences)
ft_pred_my = predict_ft(my_sentences)
# เฉลยมนุษย์ของ 3 ประโยคตัวอย่าง: 1 = pos, 2 = neg, 3 = q
truth_my = [0, 2, 3]

for s, pf, pz in zip(my_sentences, ft_pred_my, zs_pred_my):
    print(f"  [mBERT->{LABELS[pf]} | zs->{LABELS[pz]}] {s}")

print("supervised ถูก:",
      int((ft_pred_my == np.array(truth_my)).sum()), "/", len(truth_my))
print("zero-shot ถูก:",
      int((zs_pred_my == np.array(truth_my)).sum()), "/", len(truth_my))

**คำตอบข้อ 5:** ตามหลักการ (และผลที่พบจากรอบที่รัน): zero-shot ถูก **0–1 จาก 3** — มักพลาดประโยคบวกแบบแจ้งข้อมูล (ตอบ neutral) และ **ตอบคำถามไม่ได้เลย** เพราะไม่มี class คำถาม ส่วน mBERT supervised ถูก **2–3 จาก 3** (จับ "บ่น" ได้ชัดกว่า แต่ `q` ก็เป็น class ที่มันอ่อนเช่นกัน) — ตรงเป้าหมายของโมดูล: zero-shot ให้คำตอบ "ฟรีแต่หยาบ" supervised ให้ความแม่นจริงเมื่อมี label ให้เทรน ตัวเลขเป๊ะ ๆ ขึ้นกับรอบ — ให้อ่านจากผลที่พิมพ์ออกมา

**หมายเหตุ:** `truth_my` ข้างบนคือ "เฉลยมนุษย์" ของ 3 ประโยคตัวอย่าง — ถ้าใช้ประโยคของตัวเอง ให้ label เองแล้วเทียบแบบเดียวกัน

## สรุป (เฉลย)

- class ที่โมเดลอ่อนที่สุด: มักเป็น **neg** (หรือ q) — ข้อมูลน้อยที่สุด + ภาษาไทยบ่นแบบอ้อม
- error inspection เผย: การบ่นสุภาพ, คำถามไม่มี ?, ข้อความสั้นไม่มีบริบท — ตัวเลข recall เดียวไม่บอกเหล่านี้
- zero-shot พอใช้ได้เมื่อ: ไม่มี label เลย / ต้องการคำตอบแรกเร็ว ๆ — supervised จำเป็นเมื่อ: มี label หลักพันขึ้นไปและต้องการความแม่นจริง